In [ ]:
# Zolai dataset combiner (single unified dataset)
# - Scans /kaggle/input/datasets/peterpausianlian/* for known layouts
# - Also picks up Gemini / cleaner outputs under WORK_DIR/zolai_cleaner_out/ when present
# - Streams all text rows into ONE JSONL (optionally deduped)
# - Optionally exports HF DatasetDict -> save_to_disk
# Cooperates with: zolai-cleaner-v2 (Gemini JSONL), zolai-qwen-training-v2 (ZOLOAI_DATASET_SRC -> HF export)

import os
import sys
import json
import time
import hashlib
from pathlib import Path
from datetime import datetime

IS_KAGGLE = Path("/kaggle").exists()
WORK_DIR = Path(os.environ.get("ZOLOAI_WORK_DIR", "/kaggle/working" if IS_KAGGLE else str(Path.cwd())))
OUT_DIR = WORK_DIR / "zolai_combine_out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Where to scan for inputs (Kaggle datasets mount)
DATASETS_ROOT = Path(os.environ.get("ZOLOAI_DATASETS_ROOT", "/kaggle/input"))

# Output artifacts
MERGED_JSONL = OUT_DIR / "zolai_merged_all.jsonl"
COMBINE_LOG = OUT_DIR / "combine.log"
PROGRESS_LOG = OUT_DIR / "combine_progress.log"
REPORT_JSON = OUT_DIR / "combine_report.json"

# Behavior
DEDUP_EXACT = True                 # exact dedupe on cleaned text
MIN_TEXT_CHARS = 10
NORMALIZE_NFKC = True
STRIP_HTML_TAGS = True
STRIP_URLS = True

# Monitoring
USE_TQDM = True
HEARTBEAT_EVERY = 250_000          # 0 disables

# Optional: write train/val/test.jsonl from MERGED_JSONL (streaming)
WRITE_SPLITS_JSONL = True
SPLIT_VAL_FRAC = 0.01
SPLIT_TEST_FRAC = 0.01
TRAIN_JSONL = OUT_DIR / "train.jsonl"
VAL_JSONL = OUT_DIR / "val.jsonl"
TEST_JSONL = OUT_DIR / "test.jsonl"

# HF export (optional)
EXPORT_HF_DISK = True
HF_EXPORT_DIR = OUT_DIR / "zolai_hf_disk_export"
HF_VAL_FRAC = 0.01  # train/validation split for HF export only
SEED = 42

# Kaggle publish (optional)
KAGGLE_DATASET_ID = os.environ.get("ZOLOAI_KAGGLE_DATASET_ID", "peterpausianlian/zolai-master-data")
KAGGLE_VERSION_MESSAGE = "Zolai combiner v1 — unified merged dataset"
PUBLISH_KAGGLE_VERSION = False  # set True to publish
UPLOAD_STAGING = WORK_DIR / "zolai_kaggle_upload_staging"


def log_line(msg: str) -> None:
    ts = datetime.now().isoformat(timespec="seconds")
    line = f"[{ts}] {msg}"
    print(line, flush=True)
    with open(COMBINE_LOG, "a", encoding="utf-8") as f:
        f.write(line + "\n")


log_line("combine config loaded")
log_line(f"DATASETS_ROOT={DATASETS_ROOT}")
log_line(f"OUT_DIR={OUT_DIR}")


### Pipeline with other notebooks

| Step | Notebook | Output |
|------|----------|--------|
| Standard + optional Gemini | `zolai-cleaner-v2.ipynb` | `zolai_after_gemini_refine.jsonl`, optional `zolai_hf_disk_export` |
| Local Gemma (this notebook) | `zolai-cleaning-pipeline-local-gemma-2b.ipynb` | `/kaggle/working/zolai_after_local_gemma.jsonl` |
| Merge many sources | `zolai-dataset-combiner.ipynb` | `zolai_merged_all.jsonl`, `zolai_combine_out/zolai_hf_disk_export` |
| Qwen LoRA | `zolai-qwen-training-v2.ipynb` | Set `ZOLOAI_DATASET_SRC` or rely on working HF export |


In [ ]:
# 0b) Optional overrides (safe)
# All defaults live in cell 0 now. Turn APPLY_OVERRIDES=True only if you want
# to change defaults without editing cell 0.

APPLY_OVERRIDES = False

if APPLY_OVERRIDES:
    WRITE_SPLITS_JSONL = True
    SPLIT_VAL_FRAC = 0.01
    SPLIT_TEST_FRAC = 0.01

    EXPORT_HF_DISK = True
    HF_VAL_FRAC = 0.01

    KAGGLE_DATASET_ID = os.environ.get("ZOLOAI_KAGGLE_DATASET_ID", "peterpausianlian/zolai-master-data")
    KAGGLE_VERSION_MESSAGE = "Zolai combiner v1 — unified merged dataset"
    PUBLISH_KAGGLE_VERSION = False

log_line(f"overrides loaded (APPLY_OVERRIDES={APPLY_OVERRIDES})")
print("WRITE_SPLITS_JSONL:", WRITE_SPLITS_JSONL)
print("SPLIT_VAL_FRAC / SPLIT_TEST_FRAC:", SPLIT_VAL_FRAC, SPLIT_TEST_FRAC)
print("EXPORT_HF_DISK / HF_VAL_FRAC:", EXPORT_HF_DISK, HF_VAL_FRAC)
print("KAGGLE_DATASET_ID:", KAGGLE_DATASET_ID)
print("UPLOAD_STAGING:", UPLOAD_STAGING)


In [ ]:
# 1) Discover all candidate inputs under DATASETS_ROOT
# We look for:
# - single-file JSONL like zolai_train_ready_v2.jsonl
# - split JSONLs: train.jsonl / val.jsonl / test.jsonl
# - HF disk datasets: dataset_dict.json (load_from_disk layout)


def discover_sources(root: Path) -> dict:
    out = {"jsonl": [], "hf_disk": []}
    if not root.exists():
        log_line(f"WARNING: DATASETS_ROOT does not exist: {root}")
        return out

    # 1. Scan DATASETS_ROOT for all .jsonl files (recursive, fast enough for Kaggle mount)
    # We look for files matching known names or ANY .jsonl if it's in a subfolder
    for p in root.rglob("*.jsonl"):
        if p.is_file():
            out["jsonl"].append(p)

    # 2. Known HF disk dataset markers
    for p in root.rglob("dataset_dict.json"):
        if p.is_file():
            out["hf_disk"].append(p.parent)
    for p in root.rglob("dataset_info.json"):
        if p.is_file():
            # Avoid adding individual split folders (train/test) as root
            if not any(x in p.parent.name.lower() for x in ("train", "val", "test")):
                out["hf_disk"].append(p.parent)

    # 3. Cleaner / Gemma outputs in working dir (same session as zolai-cleaner-v2 or local Gemma notebook)
    for extra_root in (WORK_DIR / "zolai_cleaner_out", WORK_DIR / "zolai_combine_out"):
        if not extra_root.is_dir():
            continue
        for p in extra_root.glob("*.jsonl"):
            if p.is_file():
                out["jsonl"].append(p)

    _lg = WORK_DIR / "zolai_after_local_gemma.jsonl"
    if _lg.is_file():
        out["jsonl"].append(_lg)

    _extra = os.environ.get("ZOLOAI_EXTRA_JSONL", "").strip()
    if _extra:
        for part in _extra.split(","):
            p = Path(part.strip())
            if p.is_file():
                out["jsonl"].append(p)

    # de-dupe and sort
    out["jsonl"] = sorted(set(out["jsonl"]))
    out["hf_disk"] = sorted(set(out["hf_disk"]))
    return out


sources = discover_sources(DATASETS_ROOT)
log_line(f"found jsonl files: {len(sources['jsonl'])}")
for p in sources["jsonl"][:50]:
    print(" -", p)
if len(sources["jsonl"]) > 50:
    print(" ...")

log_line(f"found hf disk dataset roots: {len(sources['hf_disk'])}")
for p in sources["hf_disk"][:50]:
    print(" -", p)
if len(sources["hf_disk"]) > 50:
    print(" ...")


In [ ]:
# 2) Cleaning helpers (same intent as zolai-cleaner-v2)
import re
import unicodedata

_url_re = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
_tag_re = re.compile(r"<[^>]+>")
_ws_re = re.compile(r"\s+")


def clean_text(s: str) -> tuple[str, str, dict]:
    """Return (cleaned, status, metrics). status in {'KEEP','SHORT','EMPTY'}"""
    if s is None:
        return "", "EMPTY", {"html_removed": 0, "url_removed": 0, "raw_chars": 0}
    if not isinstance(s, str):
        s = str(s)

    raw = s
    if NORMALIZE_NFKC:
        s = unicodedata.normalize("NFKC", s)

    metrics = {"html_removed": 0, "url_removed": 0, "raw_chars": len(s)}

    if STRIP_HTML_TAGS:
        before = len(s)
        s = _tag_re.sub(" ", s)
        metrics["html_removed"] = max(0, before - len(s))

    if STRIP_URLS:
        before = len(s)
        s = _url_re.sub(" ", s)
        metrics["url_removed"] = max(0, before - len(s))

    s = _ws_re.sub(" ", s).strip()

    if not s:
        return "", "EMPTY", metrics
    if len(s) < MIN_TEXT_CHARS:
        return s, "SHORT", metrics
    return s, "KEEP", metrics


def text_hash(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()


In [ ]:
# 3) Stream all sources -> one merged JSONL
# Supports JSONL lines like:
# - {"text": "..."}
# - already-cleaned rows with extra keys
# - HF disk datasets with a `text` column

from collections import defaultdict

def iter_jsonl_rows(path: Path):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except Exception:
                # skip malformed lines
                yield {"__parse_error__": True, "text": ""}


def iter_hf_disk_rows(root: Path):
    from datasets import Dataset, DatasetDict, load_from_disk

    d = load_from_disk(str(root))
    if isinstance(d, DatasetDict):
        for split, ds in d.items():
            if "text" not in ds.column_names:
                continue
            for row in ds:
                yield row
        return
    if isinstance(d, Dataset):
        ds = d
        if "text" not in ds.column_names:
            return
        for row in ds:
            yield row
        return

    return


stats = defaultdict(int)
seen = set()

def heartbeat(scanned: int, saved: int, dupes: int, short: int, empty: int, t0: float):
    if not HEARTBEAT_EVERY or scanned % HEARTBEAT_EVERY != 0:
        return
    elapsed = time.time() - t0
    hb = (
        f"[{datetime.now().isoformat(timespec='seconds')}] scanned={scanned:,} saved={saved:,} "
        f"dupes={dupes:,} short={short:,} empty={empty:,} elapsed_s={elapsed:.0f}"
    )
    if USE_TQDM:
        from tqdm.auto import tqdm
        tqdm.write(hb)
    else:
        print(hb, flush=True)
    with open(PROGRESS_LOG, "a", encoding="utf-8") as pl:
        pl.write(hb + "\n")
        pl.flush()


PROGRESS_LOG.write_text(
    f"start {datetime.now().isoformat(timespec='seconds')} HEARTBEAT_EVERY={HEARTBEAT_EVERY}\n",
    encoding="utf-8",
)

log_line(f"writing merged JSONL -> {MERGED_JSONL}")

# Build a list of iterators: jsonl files + hf disk datasets
work_items = []
for p in sources["jsonl"]:
    work_items.append(("jsonl", p))
for p in sources["hf_disk"]:
    work_items.append(("hf_disk", p))

log_line(f"work items: {len(work_items)}")

# Optional tqdm wrapper over work items (coarse)
if USE_TQDM:
    from tqdm.auto import tqdm
    work_iter = tqdm(work_items, desc="sources", unit="src", mininterval=1.0, file=sys.stdout, dynamic_ncols=True)
else:
    work_iter = work_items


t0 = time.time()
with open(MERGED_JSONL, "w", encoding="utf-8") as out:
    for kind, p in work_iter:
        log_line(f"processing {kind}: {p}")
        if kind == "jsonl":
            row_iter = iter_jsonl_rows(p)
        else:
            row_iter = iter_hf_disk_rows(p)

        # Row-level tqdm is intentionally OFF (too spammy). We rely on heartbeats.
        for obj in row_iter:
            stats["scanned"] += 1
            raw = obj.get("text", "") if isinstance(obj, dict) else ""
            cleaned, status, metrics = clean_text(raw)
            stats["html_removed_chars"] += metrics.get("html_removed", 0)
            stats["url_removed_chars"] += metrics.get("url_removed", 0)

            if status == "EMPTY":
                stats["empty"] += 1
                heartbeat(stats["scanned"], stats["saved"], stats["dupes"], stats["short"], stats["empty"], t0)
                continue
            if status == "SHORT":
                stats["short"] += 1
                heartbeat(stats["scanned"], stats["saved"], stats["dupes"], stats["short"], stats["empty"], t0)
                continue

            if DEDUP_EXACT:
                h = text_hash(cleaned)
                if h in seen:
                    stats["dupes"] += 1
                    heartbeat(stats["scanned"], stats["saved"], stats["dupes"], stats["short"], stats["empty"], t0)
                    continue
                seen.add(h)

            out.write(json.dumps({"text": cleaned}, ensure_ascii=False) + "\n")
            stats["saved"] += 1
            heartbeat(stats["scanned"], stats["saved"], stats["dupes"], stats["short"], stats["empty"], t0)

elapsed = time.time() - t0
summary = {
    "scanned": int(stats["scanned"]),
    "saved": int(stats["saved"]),
    "dupes": int(stats["dupes"]),
    "short": int(stats["short"]),
    "empty": int(stats["empty"]),
    "html_removed_chars": int(stats["html_removed_chars"]),
    "url_removed_chars": int(stats["url_removed_chars"]),
    "elapsed_min": round(elapsed / 60, 2),
    "output_jsonl": str(MERGED_JSONL),
}
log_line(f"DONE merge in {summary['elapsed_min']} min | scanned={summary['scanned']:,} saved={summary['saved']:,} dupes={summary['dupes']:,} short={summary['short']:,} empty={summary['empty']:,}")
REPORT_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")
log_line(f"wrote report -> {REPORT_JSON}")


In [7]:
# 4) Optional: export to HF disk dataset (single train split, or train/val split)
if not EXPORT_HF_DISK:
    log_line("EXPORT_HF_DISK=False; skip HF export")
else:
    from datasets import Dataset, DatasetDict

    log_line("loading merged jsonl into datasets (may take a while)")
    ds = Dataset.from_json(str(MERGED_JSONL))

    if HF_VAL_FRAC and 0 < HF_VAL_FRAC < 1:
        dsd = ds.train_test_split(test_size=HF_VAL_FRAC, seed=SEED)
        d = DatasetDict({"train": dsd["train"], "validation": dsd["test"]})
        log_line(f"split train/validation with HF_VAL_FRAC={HF_VAL_FRAC}")
    else:
        d = DatasetDict({"train": ds})
        log_line("no validation split; exporting train only")

    HF_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    log_line(f"save_to_disk -> {HF_EXPORT_DIR}")
    d.save_to_disk(str(HF_EXPORT_DIR))
    log_line("HF export done")
    print("Saved:", HF_EXPORT_DIR)
    print("Set ZOLOAI_DATASET_SRC=", HF_EXPORT_DIR)


[2026-03-29T17:09:26] loading merged jsonl into datasets (may take a while)
[2026-03-29T17:09:27] split train/validation with HF_VAL_FRAC=0.01
[2026-03-29T17:09:27] save_to_disk -> /kaggle/working/zolai_combine_out/zolai_hf_disk_export


Saving the dataset (0/2 shards):   0%|          | 0/4417919 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/44626 [00:00<?, ? examples/s]

[2026-03-29T17:09:56] HF export done
Saved: /kaggle/working/zolai_combine_out/zolai_hf_disk_export
Set ZOLOAI_DATASET_SRC= /kaggle/working/zolai_combine_out/zolai_hf_disk_export


In [8]:
# 5) Optional: write train/val/test.jsonl from MERGED_JSONL (streaming, deterministic)
# This avoids loading everything into RAM.


def _bucket_from_text(text: str) -> float:
    # Deterministic pseudo-random in [0,1) from md5
    h = hashlib.md5(text.encode("utf-8")).hexdigest()[:8]
    return int(h, 16) / 16**8


if not WRITE_SPLITS_JSONL:
    log_line("WRITE_SPLITS_JSONL=False; skip split jsonl writing")
else:
    if SPLIT_VAL_FRAC < 0 or SPLIT_TEST_FRAC < 0 or (SPLIT_VAL_FRAC + SPLIT_TEST_FRAC) >= 1:
        raise ValueError("Bad split fractions. Need val>=0, test>=0, and val+test < 1")

    log_line(
        f"writing splits from {MERGED_JSONL.name} -> train/val/test with val={SPLIT_VAL_FRAC} test={SPLIT_TEST_FRAC}"
    )

    counts = {"train": 0, "val": 0, "test": 0}
    t0 = time.time()

    with open(MERGED_JSONL, "r", encoding="utf-8", errors="replace") as fin, \
        open(TRAIN_JSONL, "w", encoding="utf-8") as ftr, \
        open(VAL_JSONL, "w", encoding="utf-8") as fva, \
        open(TEST_JSONL, "w", encoding="utf-8") as fte:

        it = fin
        tqdm_mod = None
        if USE_TQDM:
            from tqdm.auto import tqdm

            tqdm_mod = tqdm
            it = tqdm(fin, desc="split", unit="line", mininterval=1.0, file=sys.stdout, dynamic_ncols=True)

        for i, line in enumerate(it, start=1):
            try:
                obj = json.loads(line)
                text = obj.get("text", "")
            except Exception:
                continue

            r = _bucket_from_text(text)
            if r < SPLIT_TEST_FRAC:
                fte.write(line)
                counts["test"] += 1
            elif r < (SPLIT_TEST_FRAC + SPLIT_VAL_FRAC):
                fva.write(line)
                counts["val"] += 1
            else:
                ftr.write(line)
                counts["train"] += 1

            if HEARTBEAT_EVERY and i % HEARTBEAT_EVERY == 0:
                hb = (
                    f"[{datetime.now().isoformat(timespec='seconds')}] split scanned={i:,} "
                    f"train={counts['train']:,} val={counts['val']:,} test={counts['test']:,} "
                    f"elapsed_s={time.time() - t0:.0f}"
                )
                if tqdm_mod is not None:
                    tqdm_mod.write(hb)
                else:
                    print(hb, flush=True)
                with open(PROGRESS_LOG, "a", encoding="utf-8") as pl:
                    pl.write(hb + "\n")
                    pl.flush()

    log_line(
        f"DONE split | train={counts['train']:,} val={counts['val']:,} test={counts['test']:,} -> {OUT_DIR}"
    )


[2026-03-29T17:11:40] writing splits from zolai_merged_all.jsonl -> train/val/test with val=0.01 test=0.01


split: 0line [00:00, ?line/s]

[2026-03-29T17:11:42] split scanned=250,000 train=244,998 val=2,486 test=2,516 elapsed_s=2
[2026-03-29T17:11:44] split scanned=500,000 train=489,982 val=4,992 test=5,026 elapsed_s=4
[2026-03-29T17:11:45] split scanned=750,000 train=734,973 val=7,446 test=7,581 elapsed_s=5
[2026-03-29T17:11:47] split scanned=1,000,000 train=979,943 val=9,947 test=10,110 elapsed_s=7
[2026-03-29T17:11:49] split scanned=1,250,000 train=1,224,871 val=12,457 test=12,672 elapsed_s=9
[2026-03-29T17:11:51] split scanned=1,500,000 train=1,469,769 val=15,013 test=15,218 elapsed_s=11
[2026-03-29T17:11:53] split scanned=1,750,000 train=1,714,663 val=17,551 test=17,786 elapsed_s=12
[2026-03-29T17:11:54] split scanned=2,000,000 train=1,959,638 val=20,097 test=20,265 elapsed_s=14
[2026-03-29T17:11:58] split scanned=2,250,000 train=2,204,645 val=22,551 test=22,804 elapsed_s=17
[2026-03-29T17:12:00] split scanned=2,500,000 train=2,449,594 val=25,072 test=25,334 elapsed_s=20
[2026-03-29T17:12:01] split scanned=2,750,000 

In [11]:
# 6) Kaggle dataset: stage + publish (mirrors zolai-cleaner-v2)
# - Stages files + writes dataset-metadata.json
# - Prints exact CLI command
# - Publishes only if PUBLISH_KAGGLE_VERSION=True
PUBLISH_KAGGLE_VERSION=True

import shutil
import subprocess

log_line("[13-kaggle] staging files for dataset version")
UPLOAD_STAGING.mkdir(parents=True, exist_ok=True)

FILES_TO_PUBLISH = [MERGED_JSONL]

# include splits if you wrote them
if WRITE_SPLITS_JSONL:
    FILES_TO_PUBLISH += [TRAIN_JSONL, VAL_JSONL, TEST_JSONL]

# include small metadata/report artifacts
FILES_TO_PUBLISH += [REPORT_JSON, COMBINE_LOG]

# include zip artifacts if you generated them (cell 7)
ZIP_RAW_PATH = WORK_DIR / "zolai_combined_dataset.zip"
ZIP_HF_PATH = WORK_DIR / "zolai_hf_disk_export.zip"
if ZIP_RAW_PATH.is_file():
    FILES_TO_PUBLISH.append(ZIP_RAW_PATH)
if ZIP_HF_PATH.is_file():
    FILES_TO_PUBLISH.append(ZIP_HF_PATH)

metadata = {
    "id": KAGGLE_DATASET_ID,
    "title": "Zolai master / cleaned corpus",
    "licenses": [{"name": "CC0-1.0"}],
}

for p in FILES_TO_PUBLISH:
    p = Path(p)
    if p.is_file():
        shutil.copy2(p, UPLOAD_STAGING / p.name)
        print("Staged:", p.name)
    else:
        print("Skip (missing):", p)

with open(UPLOAD_STAGING / "dataset-metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# Show what will be uploaded
staging_dir = Path(UPLOAD_STAGING)
message = KAGGLE_VERSION_MESSAGE
meta = staging_dir / "dataset-metadata.json"
if not staging_dir.is_dir():
    raise FileNotFoundError(f"Staging dir not found: {staging_dir}")
if not meta.is_file():
    raise FileNotFoundError(f"Missing {meta} (staging step should create it)")

print("Staging dir:", staging_dir)
print("Files:")
for p in sorted(staging_dir.iterdir()):
    if p.is_file():
        print(" -", p.name, f"({p.stat().st_size:,} bytes)")

# Try to set strict perms if token exists (may be absent in Kaggle; CLI can still work)
kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
if kaggle_json.is_file():
    os.chmod(kaggle_json, 0o600)

cmd = ["kaggle", "datasets", "version", "-p", str(staging_dir), "-m", message]
print("Command:", " ".join(cmd))

if not PUBLISH_KAGGLE_VERSION:
    print("PUBLISH_KAGGLE_VERSION is False — skip publish. Set True in cell 0 to publish.")
    log_line("[13-kaggle] staging ready (publish disabled)")
else:
    subprocess.run(cmd, check=True)
    log_line("[13-kaggle] dataset version published")


[2026-03-29T17:16:21] [13-kaggle] staging files for dataset version
Staged: zolai_merged_all.jsonl
Staged: train.jsonl
Staged: val.jsonl
Staged: test.jsonl
Staged: combine_report.json
Staged: combine.log
Staged: zolai_combined_dataset.zip
Staged: zolai_hf_disk_export.zip
Staging dir: /kaggle/working/zolai_kaggle_upload_staging
Files:
 - combine.log (7,817 bytes)
 - combine_report.json (253 bytes)
 - dataset-metadata.json (149 bytes)
 - test.jsonl (8,284,256 bytes)
 - train.jsonl (797,584,279 bytes)
 - val.jsonl (7,934,193 bytes)
 - zolai_combined_dataset.zip (468,646,428 bytes)
 - zolai_hf_disk_export.zip (254,416,022 bytes)
 - zolai_merged_all.jsonl (813,802,728 bytes)
Command: kaggle datasets version -p /kaggle/working/zolai_kaggle_upload_staging -m Zolai combiner v1 — unified merged dataset
Starting upload for file test.jsonl


100%|██████████| 7.90M/7.90M [00:01<00:00, 5.71MB/s]
  0%|          | 0.00/761M [00:00<?, ?B/s]

Upload successful: test.jsonl (8MB)
Starting upload for file train.jsonl


100%|██████████| 761M/761M [00:17<00:00, 44.6MB/s] 
  0%|          | 0.00/7.57M [00:00<?, ?B/s]

Upload successful: train.jsonl (761MB)
Starting upload for file val.jsonl


...

Upload successful: zolai_merged_all.jsonl (776MB)
Starting upload for file zolai_combined_dataset.zip


100%|██████████| 447M/447M [00:11<00:00, 41.8MB/s] 
  0%|          | 0.00/253 [00:00<?, ?B/s]

Upload successful: zolai_combined_dataset.zip (447MB)
Starting upload for file combine_report.json


100%|██████████| 253/253 [00:00<00:00, 794B/s]


Upload successful: combine_report.json (253B)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/peterpausianlian/zolai-master-data
[2026-03-29T17:17:26] [13-kaggle] dataset version published


In [10]:
# 7) Zip outputs for download + Kaggle upload
# Produces stable zip files in /kaggle/working:
# - zolai_combined_dataset.zip (raw JSONL + splits + logs)
# - zolai_hf_disk_export.zip (HF save_to_disk folder, if present)

import shutil

ZIP_RAW_NAME = "zolai_combined_dataset.zip"
ZIP_HF_NAME = "zolai_hf_disk_export.zip"
ZIP_RAW_PATH = WORK_DIR / ZIP_RAW_NAME
ZIP_HF_PATH = WORK_DIR / ZIP_HF_NAME
ZIP_STAGING = OUT_DIR / "_zip_staging"

INCLUDE_SPLITS_IN_ZIP = True
INCLUDE_HF_EXPORT_ZIP = True

log_line(f"zipping outputs -> {ZIP_RAW_PATH}")

if ZIP_STAGING.exists():
    shutil.rmtree(ZIP_STAGING)
ZIP_STAGING.mkdir(parents=True, exist_ok=True)

# Raw artifacts that are easy to consume anywhere
FILES = [MERGED_JSONL, REPORT_JSON, COMBINE_LOG]
if INCLUDE_SPLITS_IN_ZIP and WRITE_SPLITS_JSONL:
    FILES += [TRAIN_JSONL, VAL_JSONL, TEST_JSONL]

for p in FILES:
    p = Path(p)
    if p.is_file():
        shutil.copy2(p, ZIP_STAGING / p.name)
    else:
        print("Skip (missing):", p)

# Make raw zip
if ZIP_RAW_PATH.exists():
    ZIP_RAW_PATH.unlink()
shutil.make_archive(str(ZIP_RAW_PATH.with_suffix("")), "zip", root_dir=str(ZIP_STAGING))
print("Wrote:", ZIP_RAW_PATH)

# Make HF export zip (so it can be uploaded to Kaggle as a single file)
if INCLUDE_HF_EXPORT_ZIP and EXPORT_HF_DISK and Path(HF_EXPORT_DIR).is_dir():
    if ZIP_HF_PATH.exists():
        ZIP_HF_PATH.unlink()
    shutil.make_archive(
        str(ZIP_HF_PATH.with_suffix("")),
        "zip",
        root_dir=str(Path(HF_EXPORT_DIR).parent),
        base_dir=str(Path(HF_EXPORT_DIR).name),
    )
    print("Wrote:", ZIP_HF_PATH)
else:
    print("HF export zip skipped (missing HF_EXPORT_DIR or EXPORT_HF_DISK=False)")

print("Download from Kaggle files/output panel:")
print(" -", ZIP_RAW_PATH)
if ZIP_HF_PATH.exists():
    print(" -", ZIP_HF_PATH)


[2026-03-29T17:12:56] zipping outputs -> /kaggle/working/zolai_combined_dataset.zip
Wrote: /kaggle/working/zolai_combined_dataset.zip
Wrote: /kaggle/working/zolai_hf_disk_export.zip
Download from Kaggle files/output panel:
 - /kaggle/working/zolai_combined_dataset.zip
 - /kaggle/working/zolai_hf_disk_export.zip


## Notes / how to use

- This notebook is for making **one single unified dataset** instead of many separate versions.
- It scans `DATASETS_ROOT` for:
  - `zolai_train_ready_v2.jsonl`, **`zolai_after_gemini_refine.jsonl`**, **`zolai_after_local_gemma.jsonl`**
  - `train.jsonl` / `val.jsonl` / `test.jsonl`
  - Hugging Face `save_to_disk` datasets (folder contains `dataset_dict.json`)
- It also picks up **`WORK_DIR/zolai_cleaner_out/`** (outputs from **`zolai-cleaner-v2.ipynb`**) and **`WORK_DIR/zolai_after_local_gemma.jsonl`** (from **`zolai-cleaning-pipeline-local-gemma-2b.ipynb`**). Extra paths: env **`ZOLOAI_EXTRA_JSONL`** (comma-separated).
- After **`save_to_disk`**, point **`zolai-qwen-training-v2.ipynb`** at `HF_EXPORT_DIR` via **`ZOLOAI_DATASET_SRC=/kaggle/working/zolai_combine_out/zolai_hf_disk_export`** (or rely on auto-detection if that folder exists in the same Kaggle session).

### Outputs

- **Merged JSONL**: `MERGED_JSONL` (default `.../zolai_combine_out/zolai_merged_all.jsonl`)
- **Progress log**: `PROGRESS_LOG` (tail it while running)
- **HF disk export** (optional): `HF_EXPORT_DIR`

### Monitoring

In Kaggle, open a terminal and run:

```bash
tail -f /kaggle/working/zolai_combine_out/combine_progress.log
```

### Avoid double-counting

If you already have a "master" dataset like `zolai_train_ready_v2.jsonl` that itself contains the merged data, you can remove some sources by editing `sources` (discover cell) or set `DEDUP_EXACT=True` (default) to prevent duplicates.
